# Italia

Quadro nazionale del bilancio pubblico. Il notebook legge la sezione `italia` dell'export dati e mostra alcuni grafici di controllo su entrate, pressione fiscale e spesa.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SOURCE_JSON = PROJECT_ROOT / "data/export/bilancio-pubblico/source-data.json"
SECTION_DIR = PROJECT_ROOT / "data/export/bilancio-pubblico/sections"

def load_section(section_id):
    section_path = SECTION_DIR / f"{section_id}.json"
    if section_path.exists():
        payload = json.loads(section_path.read_text(encoding="utf-8"))
        return payload.get("section", payload)
    payload = json.loads(SOURCE_JSON.read_text(encoding="utf-8"))
    return payload["sections"][section_id]

def frame(rows):
    return pd.DataFrame(rows or [])

def plot_bar(data, x, y, title, top=15):
    df = data.dropna(subset=[y]).sort_values(y, ascending=False).head(top).sort_values(y)
    ax = df.plot.barh(x=x, y=y, legend=False, figsize=(9, max(4, len(df) * 0.35)))
    ax.set_title(title)
    ax.set_xlabel(y)
    ax.set_ylabel("")
    plt.tight_layout()
    return ax

def plot_line(data, x, y, title):
    df = data.dropna(subset=[x, y]).sort_values(x)
    ax = df.plot.line(x=x, y=y, marker="o", legend=False, figsize=(9, 4.8))
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(y)
    plt.tight_layout()
    return ax

italia = load_section("italia")
italia.keys()

## Entrate principali

Il grafico usa le voci principali disponibili nella sezione nazionale. I valori sono in miliardi di euro quando presenti.

In [ ]:
top_taxes = frame(italia["revenue"]["top_taxes"])
display(top_taxes[["label", "latest_year", "latest_value_mld", "source"]].head(15))
plot_bar(top_taxes.rename(columns={"latest_value_mld": "mld"}), "label", "mld", "Entrate principali disponibili")

## Pressione fiscale e contributiva

Serie storica della pressione fiscale e contributiva in percentuale del PIL.

In [ ]:
pressure = frame(italia["revenue"]["pressure_trend"])
display(pressure.tail())
plot_line(pressure, "year", "value", "Pressione fiscale e contributiva")

## Spesa pubblica per funzione

Composizione della spesa pubblica per funzione COFOG nell'ultimo anno disponibile.

In [ ]:
spending = frame(italia["spending"]["by_function"])
display(spending[["label", "year", "value_mld", "value_pil", "share_of_total"]].head(20))
plot_bar(spending, "label", "value_mld", "Spesa pubblica per funzione COFOG")